# Daily trend + cross-media priorities

This notebook uses the catalog (notebook 01) and equivalent event/media windows (notebook 02), then adds **today’s search interest**.

Open sources, no paid keys:

- [Google Trends daily RSS](https://trends.google.com/trending/rss?geo=US) across several countries
- [Wikimedia pageviews](https://wikimedia.org/api/rest_v1/) for known game and entertainment IPs
- Daily Wikipedia **product pages**, **event pages**, and Wikidata release metadata from 2026 through 2030
- Broad physical/digital event and entertainment-format registries, with confirmation labels

Franchise normalization treats Spider-Man, Spider Man, and Spiderman as the same IP. A `Spider Man 2` lookup therefore receives the `Spider-Man: Brand New Day` release window and merchandising correlation. Announced-but-unreleased titles (including TBA windows) stay in the live product set and feed event correlations. Snapshots land in `data/processed/daily/YYYY-MM-DD/`; the daily job refreshes datasets first.

In [ ]:
from datetime import date
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.daily_brief import run as run_daily_brief
from src.documents import build_retrieval_corpus
from src.load_data import load_adaptations, load_catalog, load_events
from src.priorities import (
    match_google_trends,
    priority_document,
    queries_from_text,
    retrieve_priorities,
)
from src.promote import build_plans, select_hero_products as heroes

TODAY = date.today()
print("today:", TODAY)

## Refresh today’s brief

`run_daily_brief` reuses the on-disk snapshot unless it is older than 18 hours (or you pass `refresh=True`).

In [ ]:
result = run_daily_brief(refresh=False, on=TODAY)
bundle = result["bundle"]
priorities = result["priorities"]
print("cached:" if result["cached"] else "fetched live:", result["folder"])
print(f"google trends rows: {len(bundle.get('google_trends') or [])}")
print(f"wikipedia watchlist rows: {len(bundle.get('wikipedia') or [])}")
print(f"ranked priorities: {len(priorities)}")

## Google Trends (daily RSS)

Raw trending queries, then the subset that maps onto catalog franchises.

In [ ]:
print(f"{'geo':4} {'traffic':8} query")
for row in (bundle.get("google_trends") or [])[:25]:
    print(f"{row['geo']:4} {str(row.get('traffic_label') or row['traffic']):8} {row['title']}")

catalog = load_catalog()
mapped = match_google_trends(catalog, bundle.get("google_trends") or [])
print(f"\ntrends with a catalog equivalent: {len(mapped)}")
for hit in mapped:
    titles = ", ".join(p["canonical_title"] for p in hit["products"] if p)
    print(f"  [{hit['trend']['geo']}] {hit['trend']['title']} → {titles}")

## Wikipedia attention on game IPs

Spikes vs the previous days’ median. A movie or patch can lift the related Wikipedia article before storefront demand shows up.

In [ ]:
wiki = bundle.get("wikipedia") or []
print(f"{'ratio':6} {'views':8} {'as of':10} article")
for row in wiki[:15]:
    mark = "  SPIKE" if row["spike_ratio"] >= 1.25 else ""
    print(f"{row['spike_ratio']:5.2f}x {row['views']:8} {row['as_of']:10} {row['article']}{mark}")

## Mechanism: Spider-Man movie → Spider-Man games

Even if Spider-Man is not in today’s RSS, this is the matching rule the daily job uses.

In [ ]:
example = "Spider-Man movie premiere today — new trailer trending"
print("queries:", queries_from_text(example))
print()
for row in heroes(catalog, queries_from_text(example)):
    print(f"  {row['canonical_title']}  [{row.get('product_type')}]  {row['platform']}")

## Top marketing priorities today

Score combines Google Trends matches, Wikipedia spikes, and notebook-02 event windows that are live today. A SKU that is both trending and inside an event window ranks highest.

In [ ]:
print(f"{'#':3} {'score':6} {'sources':28} product")
print("-" * 90)
for item in priorities:
    print(
        f"{item['rank']:2} {item['score']:6.1f} {','.join(item['sources'])[:28]:28} {item['canonical_title']}"
    )
    for reason in item["reasons"][:2]:
        print(f"     {reason[:110]}")
    print()

## RAG: what should we promote today?

In [ ]:
docs = [priority_document(item) for item in priorities]
events = load_events()
adaptations = load_adaptations()
plans = build_plans(events, adaptations, catalog)
corpus = build_retrieval_corpus(events, adaptations, [], plans) + docs
print(f"corpus: {len(corpus)} chunks\n")

for query in [
    f"What should we promote today {TODAY.isoformat()}?",
    "Spider-Man movie trending which games to highlight",
    "Golf FedEx Cup PGA Tour 2K merchandising",
]:
    print(f"Q: {query}")
    for score, item in retrieve_priorities(priorities, query, limit=3):
        print(f"  {score:.2f}  {item['canonical_title']}  ({', '.join(item['sources'])})")
    print()

## Keep it daily

```bash
python3 -m src.daily_brief --refresh
bash scripts/install_daily_job.sh   # macOS launchd, 08:15 every day
```

Output: `data/processed/daily/YYYY-MM-DD/{trends.json,priorities.csv,brief.md}`